In [ ]:
! pip install confluent_kafka pyshark

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [19]:
#! python -m pip install confluent-kafka
! python -m pip install pyyaml pyshark

     ---------------------------------------- 41.4/41.4 kB 1.0 MB/s eta 0:00:00
     ---------------------------------------- 4.0/4.0 MB 15.1 MB/s eta 0:00:00



[notice] A new release of pip available: 22.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup:

1. Running a Local Broker (Docker) (Kafka Raft --> KRaft)
    - Install Docker
    - create docker-compose.yaml (with the commands required to setup a broker)
    - Run the docker-compose.yaml: Open the terminal in the folder that has the yaml file and type: `docker compose up -d`
    - Verify: Our broker is now listening on localhost:9092.

2. Setting up a topic
3. Setting a producer
4. Producing some messages and sending them through the producer
5. Setting up a consumer and consuming the messages

## 1. Create a Kafka Topic
Before sending messages, we need a "folder" to store them. While many brokers create topics automatically, defining them explicitly gives us control over partitions and replication.

In [2]:
from confluent_kafka.admin import AdminClient, NewTopic

def create_topic(bootstrap_servers, topic_name, num_partitions, replication_factor):
    admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
    
    # Define topic: name, number of partitions, and replication factor
    topic = NewTopic(topic_name, num_partitions=num_partitions, replication_factor=replication_factor)
    
    fs = admin_client.create_topics([topic])

    for topic, f in fs.items():
        try:
            f.result()  # The result itself is None
            print(f"Topic '{topic}' created successfully.")
        except Exception as e:
            print(f"Failed to create topic '{topic}': {e}")

create_topic(bootstrap_servers = 'localhost:9092', topic_name = 'roy_topic1',
             num_partitions = 3, replication_factor=1)

Topic 'roy_topic1' created successfully.


To check our existing topics, we can use the `list_topics()` method from the `AdminClient`. This is essentially the programmatic version of running kafka-topics --list in your terminal.

In [3]:
def list_existing_topics(bootstrap_servers):
    # Initialize the AdminClient
    admin_client = AdminClient({'bootstrap.servers': bootstrap_servers})
    
    # Fetch cluster metadata
    metadata = admin_client.list_topics(timeout=10)
    
    print("Existing Kafka Topics:")
    print("-" * 30)
    
    # metadata.topics is a dict where keys are topic names
    for topic_name in metadata.topics:
        print(f" - {topic_name}")

# Usage
list_existing_topics(bootstrap_servers = 'localhost:9092')

Existing Kafka Topics:
------------------------------
 - roy_topic1


### 2. The Producer
- The Producer sends messages to the topic.
- It's best practice to use a "delivery report" callback to ensure the message actually arrived.

In [ ]:
from confluent_kafka import Producer

def delivery_report(err, msg):
    if err is not None:
        print(f'Message delivery failed: {err}')
    else:
        print(f'Message delivered to {msg.topic()} [{msg.partition()}]')

p = Producer({'bootstrap.servers': 'localhost:9092'})

data = ["Hello Kafka!", "Sending message 2", "Final signal"]

for val in data:
    # Trigger any available delivery report callbacks from previous produce() calls
    p.poll(0)
    p.produce('packets_stream2', val.encode('utf-8'), callback=delivery_report)

p.flush() # Wait for all messages to be delivered

Message delivered to packets_stream2 [1]
Message delivered to packets_stream2 [1]
Message delivered to packets_stream2 [0]


0

## Consumer

In [17]:
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'my-cool-app3',
    'auto.offset.reset': 'earliest' # Start from the beginning if no offset exists
}

consumer = Consumer(conf)
consumer.subscribe(['packets_stream2'])

try:
    while True:
        msg = consumer.poll(1.0) # Wait up to 1 second for a message
        if msg is None: continue
        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue

        print(f"Received message: {msg.value().decode('utf-8')}")
finally:
    consumer.close()

Received message: {"user_id": "user_19", "user_lucky_number": 838, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Blue", "value": 64569}
Received message: {"user_id": "user_09", "user_lucky_number": 507, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Green", "value": 14398}
Received message: {"user_id": "user_45", "user_lucky_number": 255, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Cyan", "value": 74226}
Received message: {"user_id": "user_84", "user_lucky_number": 15, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Yellow", "value": 87492}
Received message: {"user_id": "user_59", "user_lucky_number": 685, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Red", "value": 57139}
Received message: {"user_id": "user_33", "user_lucky_number": 370, "timestamp": "2026-03-06T17:05:30.047396", "favorite_color": "Magenta", "value": 8873}
Received message: {"user_id": "user_14", "user_lucky_number": 283, "timestamp": "2026

KeyboardInterrupt: 

In [6]:
! docker exec -it broker kafka-topics.sh --bootstrap-server localhost:9092 --list

the input device is not a TTY.  If you are using mintty, try prefixing the command with 'winpty'


In [7]:
! docker exec -i broker /opt/kafka/bin/kafka-topics.sh --bootstrap-server localhost:9092 --list

packets_stream1
packets_stream2
roy_topic1


In [9]:
! docker exec -it broker /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic <your_topic_name> --from-beginning

The system cannot find the file specified.


In [10]:
! docker exec -i broker /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic packets_stream2 --from-beginning

^C


[2026-03-06 14:48:29,761] WARN [Consumer clientId=console-consumer, groupId=console-consumer-89418] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-03-06 14:48:29,804] WARN [Consumer clientId=console-consumer, groupId=console-consumer-89418] Connection to node 1 (localhost/127.0.0.1:9092) could not be established. Node may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-03-06 14:48:30,008] WARN [Consumer clientId=console-consumer, groupId=console-consumer-89418] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Node may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-03-06 14:48:30,034] WARN [Consumer clientId=console-consumer, groupId=console-consumer-89418] Bootstrap broker localhost:9092 (id: -1 rack: null isFenced: false) disconnected (org.apache.kafka.clients.NetworkClient)


In [ ]:
docker exec -i broker /opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic packets_stream2 --from-beginning --property print.key=true --property key.separator=" | "